In [1]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import tkinter as tk
from tkinter import ttk
from PIL import Image, ImageTk
import threading
import datetime
import os
import winsound

# ── Configs ───────────────────────────────────────────────────────────────────
IMG_SIZE = 224
CHANNELS = 3
FRAME_COUNT = 20
THRESHOLD_INIT = 0.5
MODEL_PATH = "models/cctvmodel_advanced.keras"
CLIPS_DIR = "ViolenceClips"
os.makedirs(CLIPS_DIR, exist_ok=True)

WINDOW_W = 1100
WINDOW_H = 740
VIDEO_W = 1100
VIDEO_H = 618

BG_COLOR = "#1a1a1a"
BAR_COLOR = "#111111"
BTN_START = "#1e7e34"
BTN_STOP = "#c0392b"
BTN_MUTE = "#7f8c8d"
BTN_MUTE_ACT = "#e67e22"
BTN_HOVER_S = "#27ae60"
BTN_HOVER_X = "#e74c3c"
TEXT_COLOR = "#ffffff"

CAM_FPS = 30
PRE_SECONDS = 0
POST_SECONDS = 5


# ── App ───────────────────────────────────────────────────────────────────────
class ViolenceDetectorApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Violence Detection")
        self.root.geometry(f"{WINDOW_W}x{WINDOW_H}")
        self.root.resizable(False, False)
        self.root.configure(bg=BG_COLOR)

        self.analyzing = False
        self.running = True
        self.buffer = []
        self.label = ""
        self.confidence = 0.0
        self.extractor = None
        self.model = None
        self.cap = None
        self.models_ready = False
        self.predicting = False
        self.muted = False
        self.threshold = THRESHOLD_INIT

        # ── recording state ───────────────────────────────────────────────────
        self.frame_ring = []  # rolling buffer of recent raw frames
        self.ring_maxlen = int(CAM_FPS * (PRE_SECONDS + POST_SECONDS + 5))
        self.recording = False
        self.post_frames_left = 0
        self.writer = None
        self.clip_path = ""

        self._build_ui()
        self._start_camera()
        self._load_models_async()
        self._update_frame()

    # ── UI ────────────────────────────────────────────────────────────────────
    def _build_ui(self):
        self.canvas = tk.Canvas(
            self.root, width=VIDEO_W, height=VIDEO_H, bg="black", highlightthickness=0
        )
        self.canvas.pack()

        bar = tk.Frame(self.root, bg=BAR_COLOR, height=82)
        bar.pack(fill=tk.X)
        bar.pack_propagate(False)

        center = tk.Frame(bar, bg=BAR_COLOR)
        center.place(relx=0.5, rely=0.5, anchor="center")

        # Start
        self.btn_start = tk.Button(
            center,
            text="Start",
            width=11,
            height=1,
            bg="#3a3a3a",
            fg="#777777",
            font=("Helvetica", 11, "bold"),
            relief="flat",
            cursor="hand2",
            activebackground=BTN_HOVER_S,
            activeforeground=TEXT_COLOR,
            command=self._start_analysis,
            state=tk.DISABLED,
        )
        self.btn_start.pack(side=tk.LEFT, padx=10)

        # Stop
        self.btn_stop = tk.Button(
            center,
            text="Stop",
            width=11,
            height=1,
            bg="#3a3a3a",
            fg="#777777",
            font=("Helvetica", 11, "bold"),
            relief="flat",
            cursor="hand2",
            activebackground=BTN_HOVER_X,
            activeforeground=TEXT_COLOR,
            command=self._stop_analysis,
            state=tk.DISABLED,
        )
        self.btn_stop.pack(side=tk.LEFT, padx=10)

        # Mute
        self.btn_mute = tk.Button(
            center,
            text="Mute Alarm",
            width=11,
            height=1,
            bg=BTN_MUTE,
            fg=TEXT_COLOR,
            font=("Helvetica", 11, "bold"),
            relief="flat",
            cursor="hand2",
            activebackground=BTN_MUTE_ACT,
            activeforeground=TEXT_COLOR,
            command=self._toggle_mute,
        )
        self.btn_mute.pack(side=tk.LEFT, padx=10)

        # Threshold
        thr_frame = tk.Frame(center, bg=BAR_COLOR)
        thr_frame.pack(side=tk.LEFT, padx=18)

        tk.Label(
            thr_frame,
            text="Threshold",
            bg=BAR_COLOR,
            fg="#aaaaaa",
            font=("Helvetica", 9),
        ).pack()

        self.thr_var = tk.DoubleVar(value=THRESHOLD_INIT)
        self.thr_label = tk.Label(
            thr_frame,
            text=f"{THRESHOLD_INIT:.2f}",
            bg=BAR_COLOR,
            fg=TEXT_COLOR,
            font=("Helvetica", 10, "bold"),
        )
        self.thr_label.pack()

        slider = ttk.Scale(
            thr_frame,
            from_=0.1,
            to=0.9,
            orient=tk.HORIZONTAL,
            length=130,
            variable=self.thr_var,
            command=self._on_threshold_change,
        )
        slider.pack()

    # ── Model Loading ─────────────────────────────────────────────────────────
    def _load_models_async(self):
        def load():
            self.extractor = MobileNetV2(
                weights="imagenet",
                include_top=False,
                pooling="avg",
                input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS),
            )
            self.model = tf.keras.models.load_model(MODEL_PATH)
            self.models_ready = True

        threading.Thread(target=load, daemon=True).start()

    # ── Camera ────────────────────────────────────────────────────────────────
    def _start_camera(self):
        self.cap = cv2.VideoCapture(0)
        self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
        self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

    # ── Controls ──────────────────────────────────────────────────────────────
    def _start_analysis(self):
        self.analyzing = True
        self.buffer = []
        self.label = ""
        self.confidence = 0.0
        self.btn_start.config(state=tk.DISABLED, bg="#3a3a3a", fg="#777777")
        self.btn_stop.config(state=tk.NORMAL, bg=BTN_STOP, fg=TEXT_COLOR)

    def _stop_analysis(self):
        self.analyzing = False
        self.buffer = []
        self.label = ""
        self.confidence = 0.0
        self._stop_recording()
        self.btn_start.config(state=tk.NORMAL, bg=BTN_START, fg=TEXT_COLOR)
        self.btn_stop.config(state=tk.DISABLED, bg="#3a3a3a", fg="#777777")

    def _toggle_mute(self):
        self.muted = not self.muted
        if self.muted:
            self.btn_mute.config(text="Unmute Alarm", bg=BTN_MUTE_ACT)
        else:
            self.btn_mute.config(text="Mute Alarm", bg=BTN_MUTE)

    def _on_threshold_change(self, val):
        self.threshold = round(float(val), 2)
        self.thr_label.config(text=f"{self.threshold:.2f}")

    # ── Alarm ─────────────────────────────────────────────────────────────────
    def _trigger_alarm(self):
        if not self.muted:
            threading.Thread(
                target=lambda: winsound.Beep(1000, 800), daemon=True
            ).start()

    # ── Recording ─────────────────────────────────────────────────────────────
    def _start_recording(self, trigger_frame):
        if self.recording:
            # violence detected while already recording — extend post window
            self.post_frames_left = int(CAM_FPS * POST_SECONDS)
            return

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        self.clip_path = os.path.join(CLIPS_DIR, f"violence_{timestamp}.mp4")
        h, w = trigger_frame.shape[:2]
        self.writer = cv2.VideoWriter(
            self.clip_path, cv2.VideoWriter_fourcc(*"mp4v"), CAM_FPS, (w, h)
        )
        self.recording = True
        self.post_frames_left = int(CAM_FPS * POST_SECONDS)
        print(f"  Recording → {self.clip_path}")

    def _write_frame(self, frame):
        if self.writer and self.recording:
            self.writer.write(frame)

    def _stop_recording(self):
        if self.writer:
            self.writer.release()
            self.writer = None
            print(f"  Saved    → {self.clip_path}")
        self.recording = False
        self.post_frames_left = 0

    # ── Prediction ────────────────────────────────────────────────────────────
    def _predict(self, frames):
        processed = np.array(
            [
                preprocess_input(cv2.resize(f, (IMG_SIZE, IMG_SIZE)).astype(np.float32))
                for f in frames
            ]
        )
        features = self.extractor.predict(processed, verbose=0)
        prob = self.model.predict(np.expand_dims(features, axis=0), verbose=0)[0][0]
        self.confidence = prob * 100
        self.label = "Violence" if prob >= self.threshold else "Normal"

        if self.label == "Violence":
            self._trigger_alarm()
            last_frame = frames[-1]
            self.root.after(0, lambda: self._start_recording(last_frame))

        self.predicting = False

    # ── Frame Loop ────────────────────────────────────────────────────────────
    def _update_frame(self):
        if not self.running:
            return

        if (
            self.models_ready
            and self.btn_start["state"] == tk.DISABLED
            and not self.analyzing
        ):
            self.btn_start.config(state=tk.NORMAL, bg=BTN_START, fg=TEXT_COLOR)

        ret, frame = self.cap.read()
        if ret:
            # ── rolling ring buffer (for pre-event context if needed later) ──
            if len(self.frame_ring) >= self.ring_maxlen:
                self.frame_ring.pop(0)
            self.frame_ring.append(frame.copy())

            # ── write to clip if recording ────────────────────────────────────
            if self.recording:
                self._write_frame(frame.copy())
                if self.label != "Violence":
                    self.post_frames_left -= 1
                    if self.post_frames_left <= 0:
                        self._stop_recording()

            # ── collect for model ─────────────────────────────────────────────
            if self.analyzing and self.models_ready:
                self.buffer.append(frame.copy())
                if len(self.buffer) == FRAME_COUNT and not self.predicting:
                    self.predicting = True
                    frames = self.buffer.copy()
                    self.buffer = []
                    threading.Thread(
                        target=self._predict, args=(frames,), daemon=True
                    ).start()

            # ── overlay ───────────────────────────────────────────────────────
            display = frame.copy()

            if self.label:
                color = (0, 0, 220) if self.label == "Violence" else (0, 200, 0)
                cv2.putText(
                    display,
                    self.label,
                    (20, 52),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.5,
                    color,
                    3,
                    cv2.LINE_AA,
                )
                cv2.putText(
                    display,
                    f"Confidence: {self.confidence:.1f}%",
                    (20, 84),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (200, 200, 200),
                    1,
                    cv2.LINE_AA,
                )

            if self.recording:
                cv2.circle(display, (display.shape[1] - 30, 30), 10, (0, 0, 220), -1)
                cv2.putText(
                    display,
                    "REC",
                    (display.shape[1] - 65, 38),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 0, 220),
                    2,
                    cv2.LINE_AA,
                )

            if self.analyzing and not self.label:
                cv2.putText(
                    display,
                    f"Collecting {len(self.buffer)}/{FRAME_COUNT}",
                    (20, 52),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (150, 150, 150),
                    1,
                    cv2.LINE_AA,
                )

            if not self.models_ready:
                cv2.putText(
                    display,
                    "Loading model...",
                    (VIDEO_W // 2 - 140, VIDEO_H // 2),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.1,
                    (170, 170, 170),
                    2,
                    cv2.LINE_AA,
                )

            img = cv2.cvtColor(
                cv2.resize(display, (VIDEO_W, VIDEO_H)), cv2.COLOR_BGR2RGB
            )
            img = ImageTk.PhotoImage(Image.fromarray(img))
            self.canvas.create_image(0, 0, anchor=tk.NW, image=img)
            self.canvas.image = img

        self.root.after(15, self._update_frame)

    # ── Cleanup ───────────────────────────────────────────────────────────────
    def _on_close(self):
        self.running = False
        self._stop_recording()
        if self.cap:
            self.cap.release()
        self.root.destroy()


# ── Run ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    root = tk.Tk()
    app = ViolenceDetectorApp(root)
    root.protocol("WM_DELETE_WINDOW", app._on_close)
    root.mainloop()


  Recording → ViolenceClips\violence_20260406_065527.mp4
  Saved    → ViolenceClips\violence_20260406_065527.mp4
  Recording → ViolenceClips\violence_20260406_065602.mp4
  Saved    → ViolenceClips\violence_20260406_065602.mp4
